## はじめに

### [shisapost](https://www.shisapost.com/)とは

業界ニュースを自動収集し、ユーザーのビジネス文脈に基づいて影響度を分析するメール配信サービスです。

**配信内容の例:**

````
📰 Alphabet's Rise to $4 Trillion Cements Status as AI Trade Winner
https://www.bloomberg.com/news/articles/...

Alphabet（Googleの親会社）の時価総額が4兆ドルに達し、AI分野での
優位性を確立しました。同社はAI研究開発に多額の投資を行い、検索、
クラウド、広告などの主要事業でAIを活用することで収益を大きく
伸ばしています。

✅ ポジティブな影響
AlphabetのAI分野での成功は、AIを活用したコンテンツ制作の重要性を
さらに高めるでしょう。より高度なAIツールの導入が進み、あなたの
制作するコンテンツの質や効率が向上する可能性があります...

⚠️ ネガティブな影響
AI技術の急速な発展は、コンテンツ制作の現場における競争を激化させる
可能性があります。人間のクリエイターの役割や価値が相対的に低下する
リスクも考えられます...

💡 具体的な示唆・アクション
短期: 最新のAIコンテンツ生成ツールの動向を、毎週少なくとも1時間は
情報収集する習慣をつけてください。HuggingFaceのリリース情報や...

中期: 現在利用しているAIツールについて、Alphabetのような先進企業が
どのような技術を導入しているかを調査し、自身の業務に活かせそうな
部分がないか検討してください...

長期: AI倫理や著作権、データプライバシーといった、AI活用に伴う
法的・倫理的な側面に関する知識を深めるための学習プログラムに
参加することを検討してください...
````

### LLM出力の3つの品質課題

しかし、このメール生成には以下の課題があります：

| 課題 | 具体例 | ビジネスインパクト |
|------|--------|-------------------|
| **A: 根拠のない追加情報** | 本文にない「500社が採用」などを断定 | 誤情報による意思決定ミス |
| **B: こじつけ・熱量ズレ** | 無関係なニュースを「大きな影響」と過大評価 | 重要な情報が埋もれる |
| **C: アクションが抽象的** | 「情報収集する」のみで5W1Hがない | 実行不可能なアクション |

**本記事の目的**: DSPyを用いて、これらの課題を定量的に改善します。

***

## トレーニングデータの準備

DSPy最適化のために、20サンプルのトレーニングデータを用意しました。

In [1]:
#| echo: false
import json
from pathlib import Path

# データセット読み込み（コードは非表示）
DATA_DIR = Path("./data")
DATASET_PATH = DATA_DIR / "impact_analysis_training_dataset.json"

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    dataset = json.load(f)

training_data = dataset["training_data"]

In [2]:
print(f"✅ トレーニングデータ: {len(training_data)}件")
print(f"✅ ユーザー文脈: {dataset['metadata']['user_context']}")

✅ トレーニングデータ: 20件
✅ ユーザー文脈: メディア・エンターテインメント業界でAIを活用したコンテンツ制作


::: {.callout-note collapse="true"}
## サンプルデータの詳細

In [3]:
# サンプル1の表示
sample = training_data[0]
print("=" * 80)
print(f"■ タイトル: {sample['input']['title'][:60]}...")
print(f"■ 期待スコア: {sample['expected_output']['impact_score']}/5")
print(f"\n■ 期待される要約:")
print(sample['expected_output']['article_summary'][:150] + "...")
print(f"\n■ 期待される短期アクション:")
print(sample['expected_output']['short_term_action'][:150] + "...")

■ タイトル: Google、Gemini Nanoのコンテキストウィンドウを100万トークンに拡大...
■ 期待スコア: 5/5

■ 期待される要約:
GoogleがGemini Nanoのコンテキストウィンドウを100万トークンに拡大。長編脚本や小説全体を一度に処理可能になり、エッジデバイスでプライバシー重視のコンテンツ生成が実現できる。...

■ 期待される短期アクション:
HuggingFaceで公開されているGemini Nanoモデルにアクセスし、簡単なテキスト生成タスクで性能や使い勝手を具体的に把握する（30分〜半日）...


:::



***

## DSPyによるプロンプト最適化の実証

このセクションでは、上記の本番プロンプトをDSPyで最適化し、**実際にLLMに送られたプロンプトの変化**を確認します。

## DSPy実装: Writer/Judgeエージェント

ここからが本記事の主要部分です。実際にDSPyコードを実装し、プロンプト最適化の効果を検証します。

### セットアップ

In [4]:
#| echo: false
import os
from pathlib import Path
import dspy
from dspy.teleprompt import BootstrapFewShot

# .env ファイルから環境変数を読み込む
env_path = Path("../../.env")
if env_path.exists():
    with open(env_path) as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith('#'):
                key, value = line.split('=', 1)
                os.environ[key] = value.strip().strip('"').strip("'")

# OpenAI API キーを取得
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("⚠️ 警告: OPENAI_API_KEY環境変数が設定されていません")
    print(f"⚠️ .envファイル確認: {env_path.exists()}")
    raise ValueError("OPENAI_API_KEY not found")

# DSPy言語モデルの設定
lm = dspy.LM('openai/gpt-5.2', api_key=api_key)
dspy.configure(lm=lm)

print("✅ DSPyセットアップ完了:")
print(f"  - LM: OpenAI GPT-5.2")
print(f"  - Optimizer: BootstrapFewShot")
print(f"  - トレーニングデータ: {len(training_data)}サンプル")

✅ DSPyセットアップ完了:
  - LM: OpenAI GPT-5.2
  - Optimizer: BootstrapFewShot
  - トレーニングデータ: 20サンプル


### Writerエージェントの定義

In [5]:
# Writer署名: 入力→出力の型を定義
class WriterSignature(dspy.Signature):
    """ニュース記事からユーザー文脈に基づいた影響度分析を生成する"""
    
    title: str = dspy.InputField(desc="ニュース記事のタイトル")
    news_content: str = dspy.InputField(desc="ニュース記事の本文")
    user_context: str = dspy.InputField(desc="ユーザーの業界・職種")
    
    impact_score: int = dspy.OutputField(desc="影響度スコア（1-5）")
    article_summary: str = dspy.OutputField(desc="150文字前後の要約")
    positive_impact: str = dspy.OutputField(desc="ポジティブな影響の説明")
    negative_impact: str = dspy.OutputField(desc="ネガティブな影響の説明")
    short_term_action: str = dspy.OutputField(desc="短期アクション（5W1H明記）")

# Writerモジュール: ChainOfThoughtで推論プロセスを含む
class Writer(dspy.Module):
    def __init__(self):
        super().__init__()
        self.generate = dspy.ChainOfThought(WriterSignature)
    
    def forward(self, title, news_content, user_context):
        return self.generate(
            title=title,
            news_content=news_content,
            user_context=user_context
        )

print("✅ Writerモジュール定義完了")

✅ Writerモジュール定義完了


### Judgeエージェントの定義

**課題別評価Judgeの定義**

In [7]:
# A: 根拠のない追加情報 (Faithfulness)
class FaithfulnessJudge(dspy.Signature):
    """出力（要約）が入力ニュース記事に基づいているか、幻覚がないかを判定"""
    source_text: str = dspy.InputField(desc="元のニュース記事本文")
    generated_text: str = dspy.InputField(desc="生成された要約")
    
    claims: list[str] = dspy.OutputField(desc="生成文から抽出した主要な主張リスト")
    unsupported_claims: list[str] = dspy.OutputField(desc="ソースに基づく根拠が見当たらない主張")
    score: float = dspy.OutputField(desc="0.0-1.0のスコア。1.0=全て根拠あり")

# B: こじつけ・熱量ズレ (Relevance Calibration)
class RelevanceJudge(dspy.Signature):
    """ニュースとユーザー文脈の関連性を評価し、Writerがつけたスコアが妥当か判定"""
    news_content: str = dspy.InputField(desc="ニュース記事")
    user_context: str = dspy.InputField(desc="ユーザーの業界・職種")
    stated_score: int = dspy.InputField(desc="Writerがつけた影響度スコア(1-5)")
    
    actual_score: int = dspy.OutputField(desc="客観的に見た適正スコア(1-5)")
    reasoning: str = dspy.OutputField(desc="適正スコアの理由")
    
# C: アクションが抽象的 (Action Specificity)
class ActionSpecificityJudge(dspy.Signature):
    """アクションアイテムが具体的（5W1H）で実行可能かを評価"""
    action_item: str = dspy.InputField(desc="提案されたアクション")
    
    has_what: bool = dspy.OutputField(desc="何をするか明確か")
    has_who: bool = dspy.OutputField(desc="誰がやるか明確か (自分/チーム等)")
    has_when: bool = dspy.OutputField(desc="期限やタイミングが明確か")
    has_how: bool = dspy.OutputField(desc="具体的な手段・ツールが明確か")
    is_actionable: bool = dspy.OutputField(desc="明日から即実行できる具体性があるか")

# モジュールのインスタンス化
faithfulness_judge = dspy.ChainOfThought(FaithfulnessJudge)
relevance_judge = dspy.ChainOfThought(RelevanceJudge)
action_judge = dspy.ChainOfThought(ActionSpecificityJudge)

print("✅ Judgeエージェント（Faithfulness, Relevance, Action）定義完了")

# --- 統合メトリクス ---

def combined_email_quality_metric(example, prediction, trace=None):
    """3つの課題を統合した品質スコア (LLM-as-a-Judge)"""
    
    # 1. Faithfulness Metric (課題A)
    # 要約が事実に即しているか
    try:
        f_result = faithfulness_judge(
            source_text=example.news_content,
            generated_text=prediction.article_summary
        )
        faithfulness_score = f_result.score
    except:
        faithfulness_score = 0.0

    # 2. Relevance Calibration Metric (課題B)
    # スコアのズレに対するペナルティ
    try:
        r_result = relevance_judge(
            news_content=example.news_content,
            user_context=example.user_context,
            stated_score=int(prediction.impact_score)
        )
        actual = r_result.actual_score
        stated = int(prediction.impact_score)
        
        # ズレに応じて減点 (過大評価をより重く罰する例など調整可能)
        diff = abs(actual - stated)
        if diff == 0:
            relevance_score = 1.0
        elif diff == 1:
            relevance_score = 0.5
        else:
            relevance_score = 0.0
    except:
        relevance_score = 0.0

    # 3. Action Specificity Metric (課題C)
    # 5W1Hの網羅度
    try:
        a_result = action_judge(action_item=prediction.short_term_action)
        components = [
            a_result.has_what,
            a_result.has_who, 
            a_result.has_when,
            a_result.has_how,
            a_result.is_actionable
        ]
        specificity_score = sum(components) / len(components)
    except:
        specificity_score = 0.0

    # 統合スコア (重み付け)
    # 誤情報(Faithfulness)は許容できないため高めの重み
    weights = {
        "faithfulness": 0.4,
        "relevance": 0.3,
        "specificity": 0.3
    }
    
    combined = (
        faithfulness_score * weights["faithfulness"] +
        relevance_score * weights["relevance"] +
        specificity_score * weights["specificity"]
    )
    
    # デバッグ用出力 (MIPROのログで確認可能)
    if trace is not None:
        pass # DSPyの内部トレース用
        
    return combined

print("✅ 統合評価メトリクス (LLM Judge) 定義完了")

### Writer実行: 最適化前

# デモ用のサンプルを選択
demo_sample = training_data[0]

# 最適化前のWriterを実行
writer_baseline = Writer()

result_baseline = writer_baseline(
    title=demo_sample['input']['title'],
    news_content=demo_sample['input']['news_content'],
    user_context=demo_sample['input']['user_context']
)

# 最適化前のプロンプトを保存
# inspect_history()の代わりに、lm.historyを直接取得
try:
    if hasattr(lm, 'history') and lm.history:
        # 最新のメッセージを取得
        last_prompt_before = str(lm.history[-1])
    else:
        last_prompt_before = None
except Exception as e:
    last_prompt_before = None
    print(f"警告: プロンプト履歴の取得に失敗しました: {e}")

✅ Judgeエージェント（Faithfulness, Relevance, Action）定義完了
✅ 統合評価メトリクス (LLM Judge) 定義完了


In [8]:
print("✅ 最適化前のWriter実行完了")
print(f"影響度スコア: {result_baseline.impact_score}")
print(f"要約: {result_baseline.article_summary[:100]}...")

✅ 最適化前のWriter実行完了
影響度スコア: 5
要約: GoogleがGemini Nanoのコンテキストを100万トークンに拡大。脚本や小説を一度に処理でき、Hugging Faceで公開。エッジ動作にも対応し、外部送信を抑えたプライバシー重視の生成が可...


***
## Prompt Programming (MIPRO)

### Prompt Engineering から Prompt Programming へ

従来の手動でのプロンプト試行錯誤（Prompt Engineering）から、プログラムとしてプロンプトを最適化する（Prompt Programming）アプローチへ移行します。

参照: [From Prompt Tuning to Prompt Programming](https://recruit.group.gmo/engineer/jisedai/blog/from_prompt_tuning_to_prompt_programming/)

**MIPRO (Multi-Prompt Instruction Proposal Optimizer)** は、この「Prompt Programming」を具現化するDSPyのオプティマイザーです。

**アプローチの違い**:
- **Prompt Engineering**: 人間が指示を手動で書き換える
- **Prompt Programming (MIPRO)**: 
    1. データから最適な「指示（Instruction）」を自動生成・提案
    2. 最適な「数ショット例（Few-shot）」を自動選択
    3. これらを組み合わせ、スコアが最大になるものを探索

### MIPROによる命令文（Instruction）の進化

MIPROは、単に例を追加するだけでなく、**タスクの定義そのもの（Signatureの命令文）** を書き換えます。

### MIPRO実装

In [15]:
#| output: false
from dspy.teleprompt import MIPROv2

# MIPROで最適化（指示文 + Few-shot例）
print("\n🔄 MIPRO最適化を実行中...")
print("   - Prompt Programmingアプローチ: データから最適な指示と例を探索")
print("   - auto='light'モードで高速に最適解を探索")

# データセットを最大活用（全20件）
full_trainset = []
for sample in training_data:
    ex = dspy.Example(
        title=sample['input']['title'],
        news_content=sample['input']['news_content'],
        user_context=sample['input']['user_context'],
        impact_score=str(sample['expected_output']['impact_score']), # 文字列として扱う
        article_summary=sample['expected_output']['article_summary'],
        short_term_action=sample['expected_output']['short_term_action']
    ).with_inputs('title', 'news_content', 'user_context')
    full_trainset.append(ex)

# オプティマイザーの設定
# metricに先ほど定義した統合LLM-Judgeメトリクスを指定
mipro_optimizer = dspy.MIPROv2(
    metric=combined_email_quality_metric,
    auto=None, 
    num_candidates=3,     # 候補数を減らして高速化 (実験用)
    init_temperature=1.0,
    max_bootstrapped_demos=3,
    max_labeled_demos=4,
    num_threads=1,
    verbose=False
)

# コンパイル実行（最適化）
# trainsetとvalsetを分けて使用
# 実験のためデータ数を絞って高速化: train=5件, val=5件
mipro_optimized_writer = mipro_optimizer.compile(
    Writer(),
    trainset=full_trainset[5:10], # 5件のみ使用 (実験用)
    valset=full_trainset[:5], 
    num_trials=3,          # 試行回数を3回に減らして高速化 (実験用)
    minibatch=False,
)

print("✅ MIPRO最適化完了: 最適なPrompt Programが生成されました")


🔄 MIPRO最適化を実行中...
   - Prompt Programmingアプローチ: データから最適な指示と例を探索
   - auto='light'モードで高速に最適解を探索


2026/01/14 05:44:54 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2026/01/14 05:44:54 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2026/01/14 05:44:54 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=3 sets of demonstrations...


Bootstrapping set 1/3
Bootstrapping set 2/3
Bootstrapping set 3/3


  0%|          | 0/5 [00:00<?, ?it/s]/Users/toshi/dev/career/resume/main/packages/blog/.venv/lib/python3.13/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 6: Expected `Message` - serialized value may not be as expected [field_name='message', input_value=Message(content='[[ ## re...: None}, annotations=[]), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [field_name='choices', input_value=Choices(finish_reason='st...ider_specific_fields={}), input_type=Choices])
  return self.__pydantic_serializer__.to_python(
 60%|██████    | 3/5 [01:48<01:12, 36.25s/it]
2026/01/14 05:46:43 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 2: PROPOSE INSTRUCTION CANDIDATES <==
2026/01/14 05:46:43 INFO dspy.teleprompt.mipro_optimizer_v2: We will use the few-shot examples from the previous step, a generated dataset summary, a 

Bootstrapped 3 full traces after 3 examples for up to 1 rounds, amounting to 3 attempts.


...


Average Metric: 4.79 / 5 (95.8%): 100%|██████████| 5/5 [02:14<00:00, 26.80s/it] 

2026/01/14 05:55:53 INFO dspy.evaluate.evaluate: Average Metric: 4.79 / 5 (95.8%)
2026/01/14 05:55:53 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 95.8 with parameters ['Predictor 0: Instruction 0', 'Predictor 0: Few-Shot Set 2'].
2026/01/14 05:55:53 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [94.2, 97.0, 95.8]
2026/01/14 05:55:53 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 97.0
2026/01/14 05:55:53 INFO dspy.teleprompt.mipro_optimizer_v2: =======================


2026/01/14 05:55:53 INFO dspy.teleprompt.mipro_optimizer_v2: ===== Trial 4 / 3 =====



Average Metric: 4.69 / 5 (93.8%): 100%|██████████| 5/5 [02:11<00:00, 26.24s/it] 

2026/01/14 05:58:04 INFO dspy.evaluate.evaluate: Average Metric: 4.69 / 5 (93.8%)
2026/01/14 05:58:04 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 93.8 with parameters ['Predictor 0: Instruction 0', 'Predictor 0: Few-Shot Set 1'].
2026/01/14 05:58:04 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [94.2, 97.0, 95.8, 93.8]
2026/01/14 05:58:04 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 97.0
2026/01/14 05:58:04 INFO dspy.teleprompt.mipro_optimizer_v2: =======================


2026/01/14 05:58:04 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 97.0!



✅ MIPRO最適化完了: 最適なPrompt Programが生成されました


> 💡 **実行時間**: MIPROは複数のプロンプト候補を生成・評価するため、15分程度かかります。

### 最適化されたPromptの確認

実際に最適化されたプログラムを実行し、どのようなプロンプトが生成されたかを確認します。

In [22]:
#| echo: false
# テスト実行
try:
    result_mipro = mipro_optimized_writer(
        title=demo_sample['input']['title'],
        news_content=demo_sample['input']['news_content'],
        user_context=demo_sample['input']['user_context']
    )

    print(f"影響度スコア: {result_mipro.impact_score}")
    print(f"要約: {result_mipro.article_summary[:100]}...")

except Exception as e:
    print(f"実行エラー: {e}")

影響度スコア: 4
要約: GoogleはGemini Nanoのコンテキストウィンドウを100万トークンに拡大したと発表。長編脚本や小説全体を一度に処理可能。モデルはHugging Faceで公開され、開発者が利用でき、エッジ...


### 🟢 実際の最適化されたプロンプト
以下は、MIPROが自動生成・選択した実際のプロンプトです。人間が書いた元のプロンプトから、指示内容がどのように変化し、どのような例が選ばれたかを確認してください。

In [23]:
# プロンプト履歴を表示
print("=" * 80)
print("【MIPROによってプログラムされた最終プロンプト】")
print("=" * 80)

# 直近のLLM呼び出し履歴を表示
mipro_history = lm.inspect_history(n=1)

# もし履歴が表示されない場合のフォールバック（デモ表示）
if not mipro_history:
     print("\n⚠️ プロンプト履歴の直接表示に失敗しました。内部構造を表示します。")
     if hasattr(mipro_optimized_writer, 'generate') and hasattr(mipro_optimized_writer.generate, 'demos'):
        print(f"\n採用されたFew-shot数: {len(mipro_optimized_writer.generate.demos)}")
        if hasattr(mipro_optimized_writer.generate, 'extended_signature'):
             print(f"\n最適化された指示:\n{mipro_optimized_writer.generate.extended_signature.instructions}")

print("=" * 80)

【MIPROによってプログラムされた最終プロンプト】




[2026-01-14T06:00:16.131650]

System message:

Your input fields are:
1. `title` (str): ニュース記事のタイトル
2. `news_content` (str): ニュース記事の本文
3. `user_context` (str): ユーザーの業界・職種
Your output fields are:
1. `reasoning` (str): 
2. `impact_score` (int): 影響度スコア（1-5）
3. `article_summary` (str): 150文字前後の要約
4. `positive_impact` (str): ポジティブな影響の説明
5. `negative_impact` (str): ネガティブな影響の説明
6. `short_term_action` (str): 短期アクション（5W1H明記）
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## title ## ]]
{title}

[[ ## news_content ## ]]
{news_content}

[[ ## user_context ## ]]
{user_context}

[[ ## reasoning ## ]]
{reasoning}

[[ ## impact_score ## ]]
{impact_score}        # note: the value you produce must be a single int value

[[ ## article_summary ## ]]
{article_summary}

[[ ## positive_impact ## ]]
{positive_impact}

[[ ## negative_impact ## ]]
{negative_impact}

[[ ## short_term_action ## ]]
{short_term_action}

[[ ## comp

In [25]:
# 課題が発生しやすい「難易度の高いサンプル」でBefore/Afterを比較検証
# テストデータの定義（意図的に課題A, B, Cを誘発しやすい内容）
# A: 記事にない情報を幻覚しやすい（簡素な内容）
# B: 重要そうで実は無関係（スコアを間違えやすい）
# C: アクションが定まりにくい（抽象的なニュース）
problematic_sample = {
    "title": "Acme Corpが社内ログ解析にAI導入を発表",
    "news_content": "Acme Corpは本日、社内のサーバーログ解析に自社開発のAIツールを導入したと発表しました。これによりIT部門の障害対応時間が短縮される見込みです。製品としての外販予定については言及されていません。",
    "user_context": "メディア・エンターテインメント業界でAIを活用したコンテンツ制作"
}
print("🔎 検証: 難易度の高いサンプルでの比較")
print("-" * 60)
print(f"タイトル: {problematic_sample['title']}")
print(f"文脈: {problematic_sample['user_context']}")
print("-" * 60)
# 最適化前 (Baseline)
print("\n🔴 Before (Baseline):")
try:
    res_base = writer_baseline(
        title=problematic_sample['title'],
        news_content=problematic_sample['news_content'],
        user_context=problematic_sample['user_context']
    )
    print(f"Score: {res_base.impact_score}")
    print(f"Action: {res_base.short_term_action}")
    print(f"Summary: {res_base.article_summary}")
except Exception as e:
    print(f"Error: {e}")
# 最適化後 (MIPRO)
print("\n🟢 After (MIPRO Optimized):")
try:
    res_opt = mipro_optimized_writer(
        title=problematic_sample['title'],
        news_content=problematic_sample['news_content'],
        user_context=problematic_sample['user_context']
    )
    print(f"Score: {res_opt.impact_score}")
    print(f"Action: {res_opt.short_term_action}")
    print(f"Summary: {res_opt.article_summary}")
except Exception as e:
    print(f"Error: {e}")
print("-" * 60)

🔎 検証: 難易度の高いサンプルでの比較
------------------------------------------------------------
タイトル: Acme Corpが社内ログ解析にAI導入を発表
文脈: メディア・エンターテインメント業界でAIを活用したコンテンツ制作
------------------------------------------------------------

🔴 Before (Baseline):
Score: 2
Action: When: 今週中に／Who: 制作基盤担当（情シス・SRE）＋制作PM／What: 直近1〜3か月の障害ログを棚卸しし「頻出障害トップ5」と必要ログ項目（アクセス/エラー/ジョブ/配信）を定義、AI解析（既存AIOps/LLM要約でも可）の小規模PoC計画を作成／Where: 社内の監視基盤（例：SIEM/ログ管理）上で／Why: 障害対応時間短縮と制作停止リスク低減の効果を定量化するため／How: ①ログ収集範囲決定→②匿名化/権限設計→③アラート要約・原因候補提示の検証→④MTTR/検知時間で評価、の手順で2週間PoCを回す。
Summary: Acme Corpが社内サーバーログ解析に自社AIツールを導入。IT部門の障害対応時間短縮を見込むが、外販予定には触れず。AIによる運用高度化の社内活用事例。

🟢 After (MIPRO Optimized):
Score: 2
Action: Who: 情シス（SRE/インフラ）主導、制作システム担当（編集・配信）とセキュリティ/法務を巻き込み／What: 「制作に影響の大きい障害トップ10」と該当ログ種別（配信、ストレージ、レンダー、認証等）を棚卸しし、AIログ解析の適用可否（データ機微度・必要なマスキング・期待KPI）を決めるミニ調査＋小規模PoC計画を作成／When: 半日で棚卸し、1週間でPoC計画確定／Where: 既存のログ基盤（SIEM/監視ツール/クラウドログ）上で／Why: 直接の制作AIではないが、基盤停止の損失を減らす施策として投資対効果を見極めるため／How: KPIをMTTR（復旧時間）、一次切り分け時間、誤検知率、重大障害の見逃し件数に設定し、機微情報のマスキング手順とアクセス権限（最小権限）を

### 結果分析

MIPROによって生成されたプロンプトを見ると、単に例が追加されただけでなく、**タスクの指示内容（Instruction）自体が変化している**ことがわかります。これが「Prompt Programming」の効果です。データに基づいて、モデルが最も性能を発揮しやすい指示を自動的に発見しました。